# manga-lens: so sánh model (yolo11n / yolo11m)
Chạy **một model mỗi phiên** (đổi `MODEL` ở cell 1), cùng dữ liệu và cùng augment với yolo11s.
Bật trước: **Internet On**, **GPU T4 x2**. Cần Add Input = output notebook train cũ (có `yolo/`).
Dữ liệu copy sang `/kaggle/temp` nên **output chỉ có kết quả train** (nhẹ). Kiểm tra quota GPU còn đủ.

In [ ]:
MODEL = "yolo11n"   # đổi thành "yolo11m" cho lần chạy thứ hai
CFG = {
    "yolo11n": dict(batch=32, epochs=50, time=None),   # nhỏ: batch lớn, đủ 50 epoch
    "yolo11m": dict(batch=8, epochs=50, time=9.0),     # lớn: time=9 giờ -> Ultralytics tự chỉnh số epoch cho kịp
}[MODEL]
IN = "/kaggle/input/notebooks/cphggg/notebookfe55e3cee6"  # sửa nếu `!ls /kaggle/input/notebooks/*` khác
DATA = "/kaggle/temp/yolo"

In [ ]:
!pip -q install ultralytics
%cd /kaggle/working
!rm -rf manga-lens && git clone https://github.com/CPhuongg/manga-lens.git
%cd manga-lens

## Dữ liệu: copy sang /kaggle/temp (không nằm trong output) + data.yaml

In [ ]:
!ls {IN}/yolo/images
!mkdir -p /kaggle/temp && cp -r {IN}/yolo {DATA}
open("/kaggle/working/data.yaml", "w").write(f"""path: {DATA}
train: images/train
val: images/val
test: images/test
names: {{0: frame, 1: text, 2: face, 3: body}}
""")
!ls {DATA}/images/train | wc -l

## Chạy thử nhanh (5% dữ liệu, 1 epoch): bắt lỗi hết VRAM và đo tốc độ trước khi chạy dài
Xem dòng `it/s` và `GPU_mem`. Nếu `CUDA out of memory`, hạ `batch` ở cell 1.

In [ ]:
!python scripts/train.py --data /kaggle/working/data.yaml --model {MODEL}.pt --epochs 1 --fraction 0.05 --imgsz 1024 --batch {CFG['batch']} --workers 4 --name smoke_{MODEL}

## Train thật

In [ ]:
t = f"--time {CFG['time']}" if CFG["time"] else ""
!python scripts/train.py --data /kaggle/working/data.yaml --model {MODEL}.pt --epochs {CFG['epochs']} {t} --imgsz 1024 --batch {CFG['batch']} --workers 4 --name {MODEL}

## Đo trên tập test (mAP theo lớp + thời gian suy luận)

In [ ]:
import glob
W = glob.glob(f"runs/detect/runs/{MODEL}/weights/best.pt")[0]
print(W)
!python scripts/eval_det.py --weights {W} --data /kaggle/working/data.yaml --split test

## Lưu kết quả (chỉ /kaggle/working được giữ). Nhớ tải `results.csv` và `best.pt` về.

In [ ]:
!mkdir -p /kaggle/working/out && cp -r runs /kaggle/working/out/
!ls /kaggle/working/out/runs/detect/runs/{MODEL}